In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split # 데이터셋 분리 함수
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tensorflow.keras.models import Sequential, save_model, load_model
# 과적합을 줄이기위한 방법 : dropout, 배치 정규화, L2 정규화
from tensorflow.keras.layers import Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2
# 데이터 불균형 극복
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

<font size="5" color="black">ch06. 로지스틱 회귀, 분류 - 과적합 방지 (심장병 발병 예측)</font>

```
age: 환자의 나이 (년 단위)
sex: 환자의 성별 (1 = 남성, 0 = 여성)
cp (Chest Pain Type): 흉통의 종류 (0~3 또는 1~4) - 전형적 협심증(Typical angina), 비전형적 협심증(Atypical angina), 비협심증성 통증(Non-anginal pain), 무증상(Asymptomatic)
trestbps: 안정 시 혈압 (입원 시 mm Hg 기준)
chol: 혈청 콜레스테롤 수치 (mg/dl)
fbs: 공복 혈당이 120 mg/dl 초과하는지 여부 (1 = 참, 0 = 거짓)
restecg: 안정 시 심전도 결과 (0 = 정상, 1 = ST-T 파동 이상, 2 = 좌심실 비대 소견)
thalach: 최대 심박수exang: 운동 유발 협심증 여부 (1 = 있음, 0 = 없음)
oldpeak: 안정 시와 비교한 운동으로 인한 ST 분절 하강 수치
slope: 최고 운동 시 ST 분절의 기울기 (상향/평탄/하향)
ca: 형광 투시법으로 확인된 주요 혈관 수 (0~3개)
thal: 지중해빈혈 등 혈액 관련 결함 유형 (정상, 고정 결함, 가역적 결함)
target (class): 심장 질환 진단 결과 (0 = 정상/건강, 1 = 심장병 발병/혈관 협착 50% 초과)
```

# 1. 로지스틱 회귀 분석 (이진분류)
- 데이터 생성 및 전처리
    * 엑셀 - 데이터프레임 - 결측치 처리 - X, y 분리, scale 조정, 데이터셋 분리
- 모델 생성, 학습과정설정, 학습
- 모델 평가 (시각화, 평가, 교차표)
- 모델 사용 (저장, 예측)

## 1.1 데이터 생성 및 전처리

In [ ]:
# openpyxl : xlsx 파일을 읽고 쓰는 라이브러리 
%pip install openpyxl

In [ ]:
df = pd.read_excel('data/heart-disease.xlsx')

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.isin(['?']).sum()

In [ ]:
df[(df['chol']=='?')|(df['ca']=='?')|(df['hsl']=='?')]

In [ ]:
# 방법 1 : ?가 있는 행 삭제
drop_idx = df[(df['chol']=='?')|(df['ca']=='?')|(df['hsl']=='?')].index
df.drop(drop_idx)

In [ ]:
# 방법 2 : ?를 결측치로 대체 후 결측치 삭제
df = df.replace('?', np.nan)

In [ ]:
# 결측치가 포함된 데이터 추출
df[(df['chol'].isna()|df['ca'].isna()|df['hsl'].isna())]

In [ ]:
# 결측치 처리 : dropna(), fillna(), apply()
df = df.dropna(how='any').reset_index(drop=True) # 결측치가 하나의 요소라도 있을 경우, 해당 행 삭제

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
# 타겟변수 균형 확인
df['heartDisease'].value_counts()/df.shape[0]
df['heartDisease'].value_counts(normalize=True)

In [ ]:
# X, y 분리
X = df.iloc[:,:-1].values
y = df.iloc[:,-1:].values
X.shape, y.shape

In [ ]:
# X 변수의 scale 조정

scaler = StandardScaler()

scaled_X = scaler.fit_transform(X)

In [ ]:
# 데이터셋 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(scaled_X, y,
                                                    #train_size=0.8 - test_size 지정 시 자동 처리
                                                    test_size=0.2,
                                                    random_state=7, # 난수 시드(Seed)값
                                                    stratify=y # 층화 추출 : 비율을 그대로 유지하면서 데이터를 쪼개는 방식
                                                   )

In [ ]:
# 종속 변수 균형 확인
print(pd.DataFrame(y).value_counts(normalize=True))
print(pd.DataFrame(y_train).value_counts(normalize=True))
print(pd.DataFrame(y_test).value_counts(normalize=True))

## 1.2 모델 생성 & 학습과정 설정 & 학습

### 과적합을 줄이기 위한 방법
- Dropout : 훈련 시 설정한 비율(예: 0.2)만큼 뉴런을 임의로 꺼버려, 특정 뉴런에만 의존하는 '암기식 학습'을 방지
- BatchNormalization : 미니배치 단위로 평균 0, 표준편차 1로 정규화하여 학습 품질과 속도 향상
- L2 Regularization : 손실 함수에 가중치 제곱합을 추가하여 가중치($w$)가 너무 커지지 않게 제약을 걸어 과적합 방지

**BatchNormalization 파라미터 구조 (출력 노드가 32개일 때)**
    - 노드 수($N$)의 **4배**(총 128개) 파라미터가 생성

1. 학습되는 파라미터 (64개): 직접 가중치를 조정하며 최적화하는 변수
* `gamma` (32개): 데이터 폭(스케일) 조절
* `beta` (32개): 데이터 위치(이동) 조절


2. 학습되지 않는 파라미터 (64개): 학습 중 평균/분산 기록용 (실제 테스트 시 사용)
* `moving_mean` (32개): 평균의 이동 기록
* `moving_variance` (32개): 분산의 이동 기록

In [ ]:
model = Sequential()

model.add(Input(shape=(13,)))

model.add(Dense(units=52, activation='elu', 
                kernel_regularizer=l2(0.001) # l2 정규화
               ))
model.add(BatchNormalization()) # 배치 정규화
model.add(Dropout(0.4)) # 드롭아웃

# Dense >> BatchNormalization >> Dropout 순서가 가장 안정적
model.add(Dense(units=26, activation='relu', 
                kernel_regularizer=l2(0.001)))
model.add(BatchNormalization())
model.add(Dropout(0.4))

model.add(Dense(units=13, activation='elu', 
                kernel_regularizer=l2(0.001)))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Dense(units=1, activation='sigmoid'))
model.summary()

In [ ]:
from tensorflow.keras.metrics import Recall, Precision
from tensorflow.keras.optimizers import Adam

model.compile(loss='binary_crossentropy', optimizer='adam', 
                                    metrics=['accuracy', 
                                             Recall(name='recall'), # 재현율
                                             Precision(name='precision')]) # 정밀도

In [ ]:
# 클래스(0/1) 손실 가중치 계산 (학습 데이터가 불균형할 때 학습 시 50:50의 중요도로 체감하도록 손실에 가중치 부여)
from sklearn.utils.class_weight import compute_class_weight

y_train_flat = y_train.flatten() #y_train.reshape(-1)

class_weight = compute_class_weight(
                            class_weight='balanced', # 'balanced' 옵션: 전체 샘플 수 / (클래스 수 * 클래스별 샘플 수) 공식으로 가중치 자동 산출
                            classes=np.unique(y_train_flat),
                            y = y_train_flat) # 반드시 1차원
print('class_weight :', class_weight)

class_weight_dict = dict(enumerate(class_weight))
print('class_weight_dict :', class_weight_dict)

# y_train의 빈도수
values, counts = np.unique(y_train_flat, return_counts=True)
print('빈도수 : ', dict(zip(values, counts)))
print(counts[0]*class_weight_dict[0], counts[1]*class_weight_dict[1]) # 빈도가 같아짐

ratios = counts/counts.sum()
print(ratios[0]*class_weight_dict[0], ratios[1]*class_weight_dict[1])

In [ ]:
# callback : 모델이 학습하는 동안 특정 시점마다 자동으로 실행되도록 등록해두는 보조 함수
# ReduceLROnPlateau : loss이 정체될 때 학습률을 소폭 감소시켜 최적화 수렴을 돕는 콜백
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',  # 검증 손실 관찰
    factor=0.5,          # 손실 정체 시 학습률을 0.5배로 감소
    patience=5,          # 5 에포크 동안 개선이 없으면 발동
    min_lr=0.000001       # 최소 학습률 제한
)

In [ ]:
hist = model.fit(X_train, y_train,
                epochs=200,
                batch_size=100,
                validation_split=0.2,
                class_weight=class_weight_dict,
                callbacks=[reduce_lr],
                verbose=1)

## 1.3 모델 평가 (시각화, 평가, 교차표)

In [ ]:
hist.history.keys()

In [ ]:
fig, loss_ax = plt.subplots(figsize=(10,6))
loss_ax.plot(hist.history['loss'], 'y', label='train_loss')
loss_ax.plot(hist.history['val_loss'], 'r', label='val_loss')
loss_ax.set_xlabel('epoch')
loss_ax.set_ylabel('loss', rotation=0)

acc_ax = loss_ax.twinx()
acc_ax.plot(hist.history['accuracy'], 'g', label='train_accuracy')
acc_ax.plot(hist.history['val_accuracy'], 'b', label='val_accuracy')
acc_ax.plot(hist.history['val_recall'], 'orange', label='val_recall')
acc_ax.plot(hist.history['val_precision'], 'purple', label='val_precision')
acc_ax.set_ylabel('accuracy')
loss_ax.legend(loc='center right', bbox_to_anchor=(1.5, 0.6), ncol=1)
acc_ax.legend(loc='center right', bbox_to_anchor=(1.5, 0.8), ncol=1)
plt.show()

In [ ]:
loss, accuracy, recall, precision = model.evaluate(X_test, y_test)
print(f'정확도 : {accuracy*100:.2f}%')

In [ ]:
y_hat = (model.predict(X_test)>0.5).astype(int)

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_hat)

In [ ]:
f1_score = 2 * (precision * recall) / (precision + recall)

print(f"Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall   : {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-Score : {f1_score:.4f} ({f1_score*100:.2f}%)")

## 1.4 모델 사용

In [ ]:
save_model(model, 'model/05_heart.h5')

In [ ]:
model2 = load_model('model/05_heart.h5')

In [ ]:
X[0]

In [ ]:
data = [[ 63, 1, 1, 145, 233, 1, 2, 150, 0, 2.3, 3, 0, 6]]
input_data = scaler.transform(data)
(model.predict(input_data)>= 0.5).astype(int)[0][0]

# 2. 분류 분석
- 데이터 생성 및 전처리 : 타겟변수 원-핫 인코딩
    * X_train, y_train, X_test, y_test -> y_train & y_test : 원-핫 인코딩
- 모델 생성, 학습과정설정, 학습
- 모델 평가 (시각화, 평가, 교차표)
- 모델 사용 (저장, 예측)

## 2.1 데이터 생성 및 전처리

In [ ]:
Y_train = to_categorical(y_train, 2)
Y_test = to_categorical(y_test, 2)

X_train.shape, Y_train.shape, X_test.shape, Y_test.shape

## 2.2 모델 생성 & 학습과정 설정 & 학습

In [ ]:
model = Sequential()

model.add(Input(shape=(13,)))

model.add(Dense(units=52, activation='elu', 
                kernel_regularizer=l2(0.001) # l2 정규화
               ))
model.add(BatchNormalization()) # 배치 정규화
model.add(Dropout(0.4)) # 드롭아웃

# Dense >> BatchNormalization >> Dropout 순서가 가장 안정적
model.add(Dense(units=26, activation='relu', 
                kernel_regularizer=l2(0.001)))
model.add(BatchNormalization())
model.add(Dropout(0.4))

model.add(Dense(units=13, activation='elu', 
                kernel_regularizer=l2(0.001)))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Dense(units=2, activation='softmax'))
model.summary()

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', 
                                    metrics=['accuracy', 
                                             Recall(name='recall'), # 재현율
                                             Precision(name='precision')]) # 정밀도

In [ ]:
y_train_flat = y_train.flatten() #y_train.reshape(-1)

class_weight = compute_class_weight(
                            class_weight='balanced', # 'balanced' 옵션: 전체 샘플 수 / (클래스 수 * 클래스별 샘플 수) 공식으로 가중치 자동 산출
                            classes=np.unique(y_train_flat),
                            y = y_train_flat) # 반드시 1차원

class_weight_dict = dict(enumerate(class_weight))

In [ ]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.000001
)

In [ ]:
hist = model.fit(X_train, Y_train,
                epochs=200,
                validation_split=0.2,
                class_weight=class_weight_dict,
                callbacks=[reduce_lr],
                verbose=1)

## 2.3 모델 평가 (시각화, 평가, 교차표)

In [ ]:
hist.history.keys()

In [ ]:
fig, loss_ax = plt.subplots(figsize=(10,6))
loss_ax.plot(hist.history['loss'], 'y', label='train_loss')
loss_ax.plot(hist.history['val_loss'], 'r', label='val_loss')
loss_ax.set_xlabel('epoch')
loss_ax.set_ylabel('loss', rotation=0)

acc_ax = loss_ax.twinx()
acc_ax.plot(hist.history['accuracy'], 'g', label='train_accuracy')
acc_ax.plot(hist.history['val_accuracy'], 'b', label='val_accuracy')
acc_ax.plot(hist.history['val_recall'], 'orange', label='val_recall')
acc_ax.plot(hist.history['val_precision'], 'purple', label='val_precision')
acc_ax.set_ylabel('accuracy')
loss_ax.legend(loc='center right', bbox_to_anchor=(1.5, 0.6), ncol=1)
acc_ax.legend(loc='center right', bbox_to_anchor=(1.5, 0.8), ncol=1)
plt.show()

In [ ]:
loss, accuracy, recall, precision = model.evaluate(X_test, Y_test)
print(f'정확도 : {accuracy*100:.2f}%')

In [ ]:
f1_score = 2 * (precision * recall) / (precision + recall)

print(f"Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall   : {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-Score : {f1_score:.4f} ({f1_score*100:.2f}%)")

In [ ]:
# 실제값
Y_test.argmax(axis=1)

In [ ]:
# 예측값
model2.predict(X_test).argmax(axis=1)

In [ ]:
y_hat = model.predict(X_test).argmax(axis=1)
pd.crosstab(y_test.reshape(-1), y_hat, rownames=['실제값'], colnames=['예측값'])

## 2.4 모델 사용

In [ ]:
save_model(model, 'model/05_heart_1.h5')

In [ ]:
model2 = load_model('model/05_heart_1.h5')

In [ ]:
model2.predict(X_test).argmax(axis=1)

In [ ]:
data = [[ 63, 1, 1, 145, 233, 1, 2, 150, 0, 2.3, 3, 0, 6]]
input_data = scaler.transform(data)
model2.predict(input_data).argmax(axis=1)

In [ ]:
data = [[ 63,   1,   1, 145, 233,   1,   2, 150,   0,   2,   3,   0,   6],
        [ 67,   1,   4, 160, 286,   0,   0, 108,   1,   1,   2,   3,   3]]
input_data = scaler.transform(data)
model2.predict(input_data).argmax(axis=1)